<a href="https://colab.research.google.com/github/hmmnyamminji/DL/blob/main/day16_practice1_RNN_%ED%95%B4%EB%B6%80.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# RNN - 순서대로 읽고 기억하는 신경망
# RNN의 한 걸음: h_new = tanh(W_x @ x + W_h @ h_old + b)
# 은닉 상태 h = '지금까지 읽은 것의 기억'
# 같은 가중치 W_x, W_h를 매 걸음 재사용 - 가중치 공유

# 15차시 평균 풀링: 단어를 몽땅 섞어서 한 번에(순서 소멸)
# RNN: 단어를 '하나씩' 읽는다. 직전까지의 '기억(은닉 상태 h)'을 들고 다음 단어를 읽는다
# h0(백지) ─[단어1]→ h1 ─[단어2]→ h2 ─[단어3]→ h3(문장 전체의 기억)

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# 셀 2. nn.RNN
INPUT, HIDDEN = 3, 4 # INPUT: 단어1개 3차원 벡터, HIDDEN: 기억1개 4차원 벡터
rnn = nn.RNN(INPUT, HIDDEN, batch_first = True) # nn.RNN(입력차원, 은닉차원, batch_first=True): RNN 층 하나, (배치, 단어수, 차원)

x = torch.randn(1, 5, INPUT) # (1, 5, 3) 문장1, 단어 5, 3차원 벡터
out_torch, h_torch = rnn(x) # rnn은 단어 5개면 걸음(tanh)이 5번, h_torch 마지막 걸음의 기억 h 하나(1, 1, 4), out_torch 매 걸음의 기억 h를 전부 모은 것(1, 5, 4)

W_x, W_h = rnn.weight_ih_l0, rnn.weight_hh_l0 # RNN 안에는 학습된 가중치가 있다. 입력(input) → 기억(hidden) 변환표, 직전기억(hidden) → 새기억(hidden) 변환표
b_x, b_h = rnn.bias_ih_l0, rnn.bias_hh_l0 # 편향(bias)

In [ ]:
# 셀 3. 은닉 상태 = 기억 - 갱신
print("단어를 읽어갈 때마다 변하는 기억 h(4차원 벡터):")
h = torch.zeros(HIDDEN)
for t in range(5):
    h = torch.tanh(W_x @ x[0, t] + W_h @ h + b_x + b_h) # x[0, t] = 0번 문장의 t번째 단어 벡터
    # W_x(4×3)에 x[0,t](현재 단어 3차원 벡터)를 통과시켜 → 4차원 벡터로 변환: 방금 읽은 단어가 기억에 무엇을 더할지
    # W_h(4×4)에 h(4차원 벡터)를 통과 → 4차원 벡터로 : 지금까지 읽은 것들을 얼마나 유지할지
    # b_x + b_h 학습으로 정해진 상수(각 4차원 벡터)
    # tanh() : 다 더한 값(4개 숫자)을 각각 -1 - +1 사이로 눌러준다
    print(f"단어 {t+1} 읽음 → h = {[round(v, 2) for v in h.tolist()]}")

단어를 읽어갈 때마다 변하는 기억 h(4차원 벡터):
단어 1 읽음 → h = [-0.71, 0.17, -0.73, 0.75]
단어 2 읽음 → h = [0.67, -0.44, 0.55, 0.47]
단어 3 읽음 → h = [-0.85, 0.59, -0.77, 0.32]
단어 4 읽음 → h = [0.21, -0.51, -0.11, 0.87]
단어 5 읽음 → h = [-0.53, 0.29, -0.68, 0.38]


In [ ]:
# 셀 4. 순서를 바꾸면 출력이 달라진다
x_fwd = x # 단어 순서: 1,2,3,4,5
x_rev = x.flip(dims=[1]) # 단어 순서: 5,4,3,2,1

with torch.no_grad():
  _, h_fwd = rnn(x_fwd)
  _, h_rev = rnn(x_rev)

  mean_fwd = x_fwd.mean(dim=1)
  mean_rev = x_rev.mean(dim=1)
print("[같은 단어들, 순서만 뒤집기]")
print(f"평균 방식: 두 결과가 같은가? {torch.allclose(mean_fwd, mean_rev)}")
print(f"RNN 방식 : 두 결과가 같은가? {torch.allclose(h_fwd, h_rev)}")
print(f"RNN 두 기억의 차이 크기: {(h_fwd - h_rev).abs().mean().item():.4f}")

# h가 '이전 기억을 통해' 계산되므로 읽는 순서가 다르면 다른 의미의 문장으로 인식한다

[같은 단어들, 순서만 뒤집기]
평균 방식: 두 결과가 같은가? True
RNN 방식 : 두 결과가 같은가? False
RNN 두 기억의 차이 크기: 0.1783
